# 05 · Walking the tree

The direct test of top-down. A model starts at the top of the taxonomy and chooses among the children at each level.

- **strict:** one branch all the way down, like plain routing.
- **branching:** up to three branches open at each level (Graph-Constrained Policy, arXiv 2607.11954).
- **anchored:** branching, plus first choosing among retrieved entry points rather than the root (Constraint-Aware Search,
  arXiv 2607.10588), plus showing likely labels under each option (ToT-ICD, Findings of ACL 2026).

**Reduced forms.** ToT-ICD's backtracking tools are not implemented, and each text may use at most 15 model calls.

**What would count as what, written before running.**
- Expected: strict has the lowest recall, because labels on a second branch cannot be reached. This is near-certain by
  construction, so the interesting number is how much the recovery moves give back, and at what cost in calls.
- Expected from ToT-ICD: scores fall with each tier.
- Open: whether any walk beats the trained model or the flat funnel. The papers disagree, so no prediction.
- Differences count only where the 95% paired bootstrap interval excludes zero.


In [1]:
import pandas as pd

from tiered_taxonomy import experiment as ex
from tiered_taxonomy import runs
from tiered_taxonomy.config import data_root, load_config

root = data_root()
config = load_config()
seed = config["seed"]
samples = config["samples"]


def sample_for(data):
    if data.name == ex.EURLEX:
        settings = samples["eurlex"]
        return ex.sample_eurlex(
            data,
            settings["normal_documents"],
            settings["long_documents"],
            settings["long_threshold_words"],
            seed,
        )
    return ex.sample_humset(data, samples["humset"]["per_language"], seed)


def table(results):
    rows = []
    for name, result in results.items():
        scores = result["scores"]
        row = {
            "method": name,
            "labels per item": scores["labels_per_item"],
            "exact F1": round(scores["exact"]["f1"], 3),
            "exact precision": round(scores["exact"]["precision"], 3),
            "exact recall": round(scores["exact"]["recall"], 3),
        }
        row.update({f"tier {tier} F1": round(value["f1"], 3) for tier, value in scores["tiers"].items()})
        row.update({f"{band} recall": value["recall"] for band, value in scores["rarity"].items()})
        if "run" in result:
            run = result["run"]
            row.update(
                {
                    "calls per item": round(run["calls"] / run["items"], 1),
                    "cost per 1000 items ($)": round(1000 * run["cost_usd"] / run["items"], 2),
                    "malformed": run["malformed"],
                    "invalid": run["invalid"],
                }
            )
        rows.append(row)
    return pd.DataFrame(rows)

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
eurlex = ex.load_eurlex(root)
humset = ex.load_humset(root)
benches = {data.name: runs.make_bench(data, root, config) for data in (eurlex, humset)}
sets = {data.name: data for data in (eurlex, humset)}
items = {name: sample_for(data) for name, data in sets.items()}
for name, bench in benches.items():
    runs.load_cards(bench)
    runs.embed_inputs(bench, items[name])
used = ex.project_labels([*humset.train, *humset.test], humset.taxonomy)
results = {}

In [3]:
from tiered_taxonomy.metrics import micro_f1, paired_difference

for name in sets:
    bench = benches[name]
    results[name] = {
        method: runs.run_and_score(bench, method, items[name])
        for method in ("walk_strict", "walk_branching", "walk_anchored")
    }

## MultiEURLEX

In [4]:
table(results[ex.EURLEX])

,method,labels per item,exact F1,exact precision,exact recall,tier 1 F1,tier 2 F1,tier 3 F1,common recall,rare recall,unseen recall,calls per item,cost per 1000 items ($),malformed,invalid
0,walk_strict,1.36,0.018,0.051,0.011,0.176,0.061,0.027,0.0105,0.0054,0.025,4.7,0.59,0,0
1,walk_branching,4.15,0.054,0.069,0.044,0.316,0.144,0.081,0.0405,0.0753,0.100,11.1,1.35,0,0
2,walk_anchored,2.92,0.136,0.219,0.098,0.386,0.246,0.172,0.0946,0.1183,0.175,6.8,0.88,0,1


## HumSet

In [5]:
table(results[ex.HUMSET])

,method,labels per item,exact F1,exact precision,exact recall,tier 1 F1,tier 2 F1,common recall,rare recall,calls per item,cost per 1000 items ($),malformed,invalid
0,walk_strict,1.00,0.342,0.574,0.244,0.379,0.130,0.2446,0.1935,1.4,0.13,0,2
1,walk_branching,1.83,0.419,0.479,0.373,0.459,0.187,0.3738,0.3065,1.7,0.15,0,10
2,walk_anchored,1.51,0.188,0.240,0.154,0.250,0.261,0.1506,0.3710,2.2,0.22,0,11


In [6]:
differences = {}
for name in sets:
    sample = items[name]
    ids = [item.id for item in sample]
    gold = ex.gold_exact(sample)

    def get(method, name=name, ids=ids):
        return ex.load_predictions(root, name, method, ids)

    differences[name] = {
        "branching minus strict": paired_difference(
            micro_f1, gold, get("walk_branching"), get("walk_strict")
        ),
        "anchored minus branching": paired_difference(
            micro_f1, gold, get("walk_anchored"), get("walk_branching")
        ),
        "anchored minus trained": paired_difference(micro_f1, gold, get("walk_anchored"), get("trained")),
        "anchored minus funnel": paired_difference(micro_f1, gold, get("walk_anchored"), get("funnel_cards")),
    }
within = {
    method: ex.evaluate_within_project(
        humset,
        items[ex.HUMSET],
        ex.load_predictions(root, ex.HUMSET, method, [i.id for i in items[ex.HUMSET]]),
        used,
    )
    for method in results[ex.HUMSET]
}
ex.save_results(
    "05_tree_walk",
    {
        "results": results,
        "differences": differences,
        "humset_within_project": within,
        "models": config["models"],
    },
)
differences

{'eurlex': {'branching minus strict': (0.0359826176737052,
   0.029104184859254208,
   0.04315756070970799),
  'anchored minus branching': (0.08185161746443219,
   0.0699975896498094,
   0.09498133844277841),
  'anchored minus trained': (-0.28432423406123,
   -0.302869294040499,
   -0.26541921639648525),
  'anchored minus funnel': (-0.029830068665233633,
   -0.04266390645162819,
   -0.017131714664028956)},
 'humset': {'branching minus strict': (0.07523460000119747,
   0.06301118150724558,
   0.0872657718687419),
  'anchored minus branching': (-0.23115318302839394,
   -0.2469968101265257,
   -0.2152278050330426),
  'anchored minus trained': (-0.3536161754510638,
   -0.37210992184774666,
   -0.33673706180593743),
  'anchored minus funnel': (-0.20293541741818094,
   -0.21900686668530486,
   -0.1869325207820782)}}